# YOLOv8-pose fine-tune on Amateur Drawings Dataset

Fine-tunes YOLOv8n-pose on hand-drawn human figures.

**Uses pre-prepared data from Google Drive** (upload `training/data.zip` to `MyDrive/yolo_data/data.zip`).

**Before running:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU found — switch runtime to GPU')

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics

## 2. Load data from Google Drive

Upload `training/data.zip` (~930 MB) to your Google Drive at:
`MyDrive/yolo_data/data.zip`

The zip contains:
- `images/train/` — 3000 training images
- `images/val/`   — 500 validation images
- `labels/train/` — pre-generated YOLO pose labels
- `labels/val/`
- `dataset.yaml`

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, shutil

drive.mount('/content/drive')

BASE    = Path('/content/drawings')
ZIP_SRC = Path('/content/drive/MyDrive/yolo_data/data.zip')
BASE.mkdir(exist_ok=True)

print('Copying zip from Drive...')
shutil.copy(ZIP_SRC, '/content/data.zip')
print('Extracting...')
with zipfile.ZipFile('/content/data.zip') as z:
    z.extractall(BASE)

train_imgs = sorted((BASE / 'images/train').glob('*.png')) + sorted((BASE / 'images/train').glob('*.jpg'))
val_imgs   = sorted((BASE / 'images/val').glob('*.png'))   + sorted((BASE / 'images/val').glob('*.jpg'))
print(f'train images : {len(train_imgs)}')
print(f'val   images : {len(val_imgs)}')
print('Done — labels are pre-generated, skip to section 3 (Preview) or 4 (Train)')

## 3. Preview training data with keypoints

In [ ]:
import random, cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

BASE = Path('/content/drawings')

SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

def draw_yolo_labels(img_path, label_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for line in label_path.read_text().strip().splitlines():
        vals = list(map(float, line.split()))
        if len(vals) < 5 + 17 * 3:
            continue
        cx, cy, bw, bh = vals[1], vals[2], vals[3], vals[4]
        x1, y1 = int((cx - bw/2)*w), int((cy - bh/2)*h)
        x2, y2 = int((cx + bw/2)*w), int((cy + bh/2)*h)
        cv2.rectangle(img, (x1,y1), (x2,y2), (200,200,200), 1)
        pts = [(int(vals[5+k*3]*w), int(vals[5+k*3+1]*h), vals[5+k*3+2]) for k in range(17)]
        for j1, j2 in SKELETON:
            if pts[j1][2] > 0 and pts[j2][2] > 0:
                cv2.line(img, pts[j1][:2], pts[j2][:2], (80,200,80), 2)
        for px, py, v in pts:
            if v > 0:
                cv2.circle(img, (px, py), 5, (255,80,80), -1)
    return img

train_imgs = sorted((BASE / 'images/train').glob('*.png')) + sorted((BASE / 'images/train').glob('*.jpg'))
label_dir  = BASE / 'labels/train'

random.seed(42)
sample = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for ax, ip in zip(axes, sample):
    lp = label_dir / (ip.stem + '.txt')
    if not lp.exists():
        ax.text(0.5, 0.5, 'label missing', ha='center')
        ax.axis('off')
        continue
    ax.imshow(draw_yolo_labels(ip, lp))
    n = len(lp.read_text().strip().splitlines())
    ax.set_title(f'{ip.name}  ({n} ann)', fontsize=7)
    ax.axis('off')
for ax in axes[len(sample):]:
    ax.axis('off')
plt.suptitle('YOLO pose labels — training set preview', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Train

In [ ]:
# Ensure flip_idx is present — re-enables horizontal flip augmentation
import re as _re
_yt = (BASE / "dataset.yaml").read_text()
if "flip_idx" not in _yt:
    _yt = _yt.rstrip() + "
flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
"
    (BASE / "dataset.yaml").write_text(_yt)
    print("Added flip_idx to dataset.yaml")

from ultralytics import YOLO
from pathlib import Path

BASE      = Path('/content/drawings')
YAML_PATH = BASE / 'dataset.yaml'
# Fix path to absolute so YOLO finds images regardless of working directory
import re as _re
_yt = YAML_PATH.read_text()
_yt = _re.sub(r'(?m)^path:.*', f'path: {BASE}', _yt)
YAML_PATH.write_text(_yt)

EPOCHS = 50   # @param {type:"integer"}
BATCH  = 16   # @param {type:"integer"}
IMGSZ  = 640  # @param {type:"integer"}

model   = YOLO('yolov8n-pose.pt')
results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    workers=2,
    project='/content/runs',
    name='drawn_humanoid_pose',
    exist_ok=True,
    patience=15,
    save_period=10,
    plots=True,
)
print('Best model:', results.save_dir / 'weights/best.pt')

## 5. Export to ONNX (for Flutter / mobile)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

best_pt = Path('/content/runs/drawn_humanoid_pose/weights/best.pt')
m = YOLO(str(best_pt))
m.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True,
    dynamic=False,
)
onnx_path = best_pt.with_suffix('.onnx')
print(f'ONNX saved: {onnx_path}  ({onnx_path.stat().st_size // 1024 // 1024} MB)')

## 6. Download trained model

In [ ]:
import shutil
from google.colab import files

best_pt   = '/content/runs/drawn_humanoid_pose/weights/best.pt'
best_onnx = '/content/runs/drawn_humanoid_pose/weights/best.onnx'

shutil.copy(best_pt,   '/content/drawn_humanoid_pose.pt')
shutil.copy(best_onnx, '/content/drawn_humanoid_pose.onnx')

files.download('/content/drawn_humanoid_pose.pt')
files.download('/content/drawn_humanoid_pose.onnx')
print('Downloaded: drawn_humanoid_pose.pt  +  drawn_humanoid_pose.onnx')

In [ ]:
# Optional: save to Google Drive so it survives session reset
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive', force_remount=False)
out_dir = Path('/content/drive/MyDrive/yolo_models')
out_dir.mkdir(parents=True, exist_ok=True)

shutil.copy('/content/drawn_humanoid_pose.pt',   out_dir / 'drawn_humanoid_pose.pt')
shutil.copy('/content/drawn_humanoid_pose.onnx', out_dir / 'drawn_humanoid_pose.onnx')
print(f'Saved to Drive: {out_dir}')

## 7. Test on validation images

In [ ]:
import random, cv2, numpy as np, matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path

BASE    = Path('/content/drawings')
val_dir = BASE / 'images/val'

SKELETON_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

model  = YOLO('/content/runs/drawn_humanoid_pose/weights/best.pt')
imgs   = sorted(val_dir.glob('*.png')) + sorted(val_dir.glob('*.jpg'))
sample = random.sample(imgs, min(6, len(imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample):
    img_bgr = cv2.imread(str(img_path))
    results = model(img_bgr, conf=0.01, verbose=False)
    vis     = img_bgr.copy()
    detected = False
    for r in results:
        if r.keypoints is None or len(r.keypoints.xy) == 0:
            continue
        kpts = r.keypoints.xy[0].cpu().numpy()
        if not np.any(kpts > 0):
            continue
        detected = True
        pts = [(int(kpts[k][0]), int(kpts[k][1])) for k in range(17)]
        for j1, j2 in SKELETON_PAIRS:
            if pts[j1] != (0,0) and pts[j2] != (0,0):
                cv2.line(vis, pts[j1], pts[j2], (80,200,80), 2)
        for px, py in pts:
            if (px, py) != (0, 0):
                cv2.circle(vis, (px, py), 5, (0,80,255), -1)
        conf_val = float(r.boxes.conf[0]) if r.boxes is not None and len(r.boxes.conf) else 0
        ax.set_title(f'{img_path.name}\nconf={conf_val:.2f}', fontsize=7)
        break
    if not detected:
        ax.set_title(f'{img_path.name}\nno detection', fontsize=7)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.axis('off')

for ax in axes[len(sample):]:
    ax.axis('off')

plt.suptitle('drawn_humanoid_pose.pt — validation set', fontsize=13)
plt.tight_layout()
plt.show()